# CART_colab

In [1]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.fpm import FPGrowth

In [2]:
print("🚀 Starting Dataproc Serverless session via Spark Connect...")

# Create the session (the first execution might take 1-2 minutes
# to provision the serverless infrastructure in europe-west8)
spark = DataprocSparkSession.builder.getOrCreate()

print("✅ Spark Session successfully created and ready for use!")

🚀 Starting Dataproc Serverless session via Spark Connect...


████████████████████████████▎                                                   

✅ Spark Session successfully created and ready for use!


In [3]:
# Direct pointer to the cleaned dataset file within your bucket
bucket_uri = "gs://ccbd-20260619-danieletambone-bucket/online_retail_clean.csv"

print(f"📥 Loading dataset from {bucket_uri}...")

# Read the CSV file into a distributed Spark DataFrame
df_spark = spark.read.csv(
    bucket_uri,
    header=True,
    inferSchema=True
)

# Print the schema and record count to verify the data loading process
df_spark.printSchema()

df_spark.createOrReplaceTempView("retail_data")

print(f"✅ Rows successfully loaded: {df_spark.count()}")

📥 Loading dataset from gs://ccbd-20260619-danieletambone-bucket/online_retail_clean.csv...
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



  0%|           0/3 Tasks

✅ Rows successfully loaded: 530104


## Query 1

### Spark SQL

In [4]:
print("📊 Executing Query 1 (Revenue & Volume Trends) with Spark SQL...")

query_1_sql = spark.sql("""
    SELECT
        year(InvoiceDate) AS year,
        month(InvoiceDate) AS month,
        COUNT(DISTINCT InvoiceNo) AS total_orders,
        ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue
    FROM retail_data
    GROUP BY year, month
    ORDER BY year, month
""")
query_1_sql.show(10)

📊 Executing Query 1 (Revenue & Volume Trends) with Spark SQL...


  0%|           0/1 Tasks

+----+-----+------------+-------------+
|year|month|total_orders|total_revenue|
+----+-----+------------+-------------+
|2010|   12|        1559|    823746.14|
|2011|    1|        1086|    691364.56|
|2011|    2|        1100|    523631.89|
|2011|    3|        1454|    717639.36|
|2011|    4|        1246|    537808.62|
|2011|    5|        1681|    770536.02|
|2011|    6|        1533|     761739.9|
|2011|    7|        1475|    719221.19|
|2011|    8|        1361|    759138.38|
|2011|    9|        1837|   1058590.17|
+----+-----+------------+-------------+
only showing top 10 rows


### PySpark Operators

In [5]:
print("📊 Executing Query 1 with PySpark Operators...")

query_1_ops = df_spark.groupBy(
    F.year("InvoiceDate").alias("year"),
    F.month("InvoiceDate").alias("month")
).agg(
    F.countDistinct("InvoiceNo").alias("total_orders"),
    F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("total_revenue")
).orderBy("year", "month")

query_1_ops.show(10)

📊 Executing Query 1 with PySpark Operators...


  0%|           0/6 Tasks

+----+-----+------------+-------------+
|year|month|total_orders|total_revenue|
+----+-----+------------+-------------+
|2010|   12|        1559|    823746.14|
|2011|    1|        1086|    691364.56|
|2011|    2|        1100|    523631.89|
|2011|    3|        1454|    717639.36|
|2011|    4|        1246|    537808.62|
|2011|    5|        1681|    770536.02|
|2011|    6|        1533|     761739.9|
|2011|    7|        1475|    719221.19|
|2011|    8|        1361|    759138.38|
|2011|    9|        1837|   1058590.17|
+----+-----+------------+-------------+
only showing top 10 rows


## Query 2

### Spark SQL

In [6]:
print("📊 Executing Query 2 (Sales Geography) with Spark SQL...")

query_2_sql = spark.sql("""
    SELECT
        Country AS country,
        COUNT(DISTINCT InvoiceNo) AS total_orders,
        ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue,
        ROUND(SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo), 2) AS average_order_value
    FROM retail_data
    GROUP BY Country
    ORDER BY total_revenue DESC
""")
query_2_sql.show(10)

📊 Executing Query 2 (Sales Geography) with Spark SQL...


  0%|           0/6 Tasks

+--------------+------------+-------------+-------------------+
|       country|total_orders|total_revenue|average_order_value|
+--------------+------------+-------------+-------------------+
|United Kingdom|       18019|   9025222.08|             500.87|
|   Netherlands|          94|    285446.34|            3036.66|
|          EIRE|         288|    283453.96|             984.22|
|       Germany|         457|    228867.14|              500.8|
|        France|         392|    209715.11|             534.99|
|     Australia|          57|    138521.31|             2430.2|
|         Spain|          90|     61577.11|             684.19|
|   Switzerland|          54|      57089.9|            1057.22|
|       Belgium|          98|     41196.34|             420.37|
|        Sweden|          36|     38378.33|            1066.06|
+--------------+------------+-------------+-------------------+
only showing top 10 rows


### PySpark Operators

In [7]:
print("📊 Executing Query 2 (Sales Geography) with PySpark Operators...")

query_2_ops = df_spark.groupBy("Country").agg(
    F.countDistinct("InvoiceNo").alias("total_orders"),
    F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("total_revenue"),
    F.round(
        F.sum(F.col("Quantity") * F.col("UnitPrice")) / F.countDistinct("InvoiceNo"), 2
    ).alias("average_order_value")
).withColumnRenamed("Country", "country") \
 .orderBy(F.col("total_revenue").desc())

query_2_ops.show(10)

📊 Executing Query 2 (Sales Geography) with PySpark Operators...


  0%|           0/6 Tasks

+--------------+------------+-------------+-------------------+
|       country|total_orders|total_revenue|average_order_value|
+--------------+------------+-------------+-------------------+
|United Kingdom|       18019|   9025222.08|             500.87|
|   Netherlands|          94|    285446.34|            3036.66|
|          EIRE|         288|    283453.96|             984.22|
|       Germany|         457|    228867.14|              500.8|
|        France|         392|    209715.11|             534.99|
|     Australia|          57|    138521.31|             2430.2|
|         Spain|          90|     61577.11|             684.19|
|   Switzerland|          54|      57089.9|            1057.22|
|       Belgium|          98|     41196.34|             420.37|
|        Sweden|          36|     38378.33|            1066.06|
+--------------+------------+-------------+-------------------+
only showing top 10 rows


## Query 3

### Spark SQL

In [8]:
print("📊 Executing Query 3 (Time Brackets) with Spark SQL...")

query_3_sql = spark.sql("""
    SELECT
        date_format(InvoiceDate, 'EEEE') AS day_of_week,
        CASE
            WHEN hour(InvoiceDate) BETWEEN 6 AND 12 THEN '1 - Morning (06:00 am - 12:59 pm)'
            WHEN hour(InvoiceDate) BETWEEN 13 AND 17 THEN '2 - Afternoon (01:00 pm - 05:59 pm)'
            WHEN hour(InvoiceDate) BETWEEN 18 AND 22 THEN '3 - Evening (06:00 pm - 10:59 pm)'
            ELSE '4 - Night (11:00 pm - 05:59 am)'
        END AS time_bracket,
        COUNT(DISTINCT InvoiceNo) AS total_orders
    FROM retail_data
    GROUP BY day_of_week, time_bracket
    ORDER BY total_orders DESC
""")
query_3_sql.show(10)

📊 Executing Query 3 (Time Brackets) with Spark SQL...


  0%|           0/3 Tasks

+-----------+--------------------+------------+
|day_of_week|        time_bracket|total_orders|
+-----------+--------------------+------------+
|   Thursday|2 - Afternoon (01...|        2032|
|  Wednesday|1 - Morning (06:0...|        1932|
|   Thursday|1 - Morning (06:0...|        1884|
|    Tuesday|1 - Morning (06:0...|        1838|
|  Wednesday|2 - Afternoon (01...|        1755|
|     Friday|1 - Morning (06:0...|        1750|
|    Tuesday|2 - Afternoon (01...|        1707|
|     Monday|1 - Morning (06:0...|        1574|
|     Monday|2 - Afternoon (01...|        1550|
|     Friday|2 - Afternoon (01...|        1378|
+-----------+--------------------+------------+
only showing top 10 rows


### PySpark Operators

In [9]:
print("📊 Executing Query 3 (Time Brackets) with PySpark Operators...")

query_3_ops = df_spark.withColumn("day_of_week", F.date_format("InvoiceDate", "EEEE")) \
    .withColumn("time_bracket",
        F.when((F.hour("InvoiceDate") >= 6) & (F.hour("InvoiceDate") <= 12), '1 - Morning (06:00 am - 12:59 pm)')
         .when((F.hour("InvoiceDate") >= 13) & (F.hour("InvoiceDate") <= 17), '2 - Afternoon (01:00 pm - 05:59 pm)')
         .when((F.hour("InvoiceDate") >= 18) & (F.hour("InvoiceDate") <= 22), '3 - Evening (06:00 pm - 10:59 pm)')
         .otherwise('4 - Night (11:00 pm - 05:59 am)')
    ) \
    .groupBy("day_of_week", "time_bracket") \
    .agg(F.countDistinct("InvoiceNo").alias("total_orders")) \
    .orderBy(F.col("total_orders").desc())

query_3_ops.show(10)

📊 Executing Query 3 (Time Brackets) with PySpark Operators...


  0%|           0/6 Tasks

+-----------+--------------------+------------+
|day_of_week|        time_bracket|total_orders|
+-----------+--------------------+------------+
|   Thursday|2 - Afternoon (01...|        2032|
|  Wednesday|1 - Morning (06:0...|        1932|
|   Thursday|1 - Morning (06:0...|        1884|
|    Tuesday|1 - Morning (06:0...|        1838|
|  Wednesday|2 - Afternoon (01...|        1755|
|     Friday|1 - Morning (06:0...|        1750|
|    Tuesday|2 - Afternoon (01...|        1707|
|     Monday|1 - Morning (06:0...|        1574|
|     Monday|2 - Afternoon (01...|        1550|
|     Friday|2 - Afternoon (01...|        1378|
+-----------+--------------------+------------+
only showing top 10 rows


## Query 4

### Spark SQL

In [10]:
print("📊 Executing Query 4 (RFM) with Spark SQL...")

query_4_sql = spark.sql("""
    WITH reference_date AS (
        SELECT MAX(to_date(InvoiceDate)) AS max_date FROM retail_data
    ),
    rfm_base AS (
        SELECT
            CustomerID,
            datediff((SELECT max_date FROM reference_date), MAX(to_date(InvoiceDate))) AS Recency,
            COUNT(DISTINCT InvoiceNo) AS Frequency,
            ROUND(SUM(Quantity * UnitPrice), 2) AS Monetary
        FROM retail_data
        WHERE CustomerID IS NOT NULL AND CustomerID != 0
        GROUP BY CustomerID
    )
    SELECT * FROM rfm_base
    ORDER BY Monetary DESC, Frequency DESC
""")
query_4_sql.show(10)

📊 Executing Query 4 (RFM) with Spark SQL...


  0%|           0/3 Tasks

+----------+-------+---------+---------+
|CustomerID|Recency|Frequency| Monetary|
+----------+-------+---------+---------+
|     14646|      1|       73|280206.02|
|     18102|      0|       60| 259657.3|
|     17450|      8|       46|194550.79|
|     16446|      0|        2| 168472.5|
|     14911|      1|      201|143825.06|
|     12415|     24|       21|124914.53|
|     14156|      9|       55|117379.63|
|     17511|      2|       31| 91062.38|
|     16029|     38|       63| 81024.84|
|     12346|    325|        1|  77183.6|
+----------+-------+---------+---------+
only showing top 10 rows


### PySpark Operators

In [11]:
print("📊 Executing Query 4 (RFM) with PySpark Operators...")

# Extract scalar max date to avoid heavy cross-joins in Spark
max_date_val = df_spark.select(F.max(F.to_date("InvoiceDate"))).collect()[0][0]

query_4_ops = df_spark.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .groupBy("CustomerID") \
    .agg(
        F.datediff(F.lit(max_date_val), F.max(F.to_date("InvoiceDate"))).alias("Recency"),
        F.countDistinct("InvoiceNo").alias("Frequency"),
        F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("Monetary")
    ).orderBy(F.col("Monetary").desc(), F.col("Frequency").desc())

query_4_ops.show(10)

📊 Executing Query 4 (RFM) with PySpark Operators...


  0%|           0/3 Tasks

  0%|           0/6 Tasks

+----------+-------+---------+---------+
|CustomerID|Recency|Frequency| Monetary|
+----------+-------+---------+---------+
|     14646|      1|       73|280206.02|
|     18102|      0|       60| 259657.3|
|     17450|      8|       46|194550.79|
|     16446|      0|        2| 168472.5|
|     14911|      1|      201|143825.06|
|     12415|     24|       21|124914.53|
|     14156|      9|       55|117379.63|
|     17511|      2|       31| 91062.38|
|     16029|     38|       63| 81024.84|
|     12346|    325|        1|  77183.6|
+----------+-------+---------+---------+
only showing top 10 rows


## Query 5

### Spark SQL

In [12]:
print("📊 Executing Query 5 (Cohorts) with Spark SQL...")

query_5_sql = spark.sql("""
    WITH customer_cohort AS (
        SELECT
            CustomerID,
            date_trunc('month', MIN(InvoiceDate)) AS cohort_month
        FROM retail_data
        WHERE CustomerID IS NOT NULL AND CustomerID != 0
        GROUP BY CustomerID
    ),
    customer_purchases AS (
        SELECT DISTINCT
            CustomerID,
            date_trunc('month', InvoiceDate) AS purchase_month
        FROM retail_data
        WHERE CustomerID IS NOT NULL AND CustomerID != 0
    )
    SELECT
        date_format(cc.cohort_month, 'yyyy-MM') AS acquisition_month,
        CAST(months_between(cp.purchase_month, cc.cohort_month) AS INT) AS retention_month_index,
        COUNT(DISTINCT cp.CustomerID) AS active_customers
    FROM customer_cohort cc
    INNER JOIN customer_purchases cp ON cc.CustomerID = cp.CustomerID
    GROUP BY acquisition_month, retention_month_index
    ORDER BY acquisition_month ASC, retention_month_index ASC
""")
query_5_sql.show(10)

📊 Executing Query 5 (Cohorts) with Spark SQL...


  0%|           0/4 Tasks

+-----------------+---------------------+----------------+
|acquisition_month|retention_month_index|active_customers|
+-----------------+---------------------+----------------+
|          2010-12|                    0|             885|
|          2010-12|                    1|             324|
|          2010-12|                    2|             286|
|          2010-12|                    3|             340|
|          2010-12|                    4|             321|
|          2010-12|                    5|             352|
|          2010-12|                    6|             321|
|          2010-12|                    7|             309|
|          2010-12|                    8|             313|
|          2010-12|                    9|             350|
+-----------------+---------------------+----------------+
only showing top 10 rows


### PySpark Operators

In [13]:
print("📊 Executing Query 5 (Cohorts) with PySpark Operators...")

# Sub-Table 1: Acquisition Cohort
cohort_df = df_spark.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .groupBy("CustomerID") \
    .agg(F.date_trunc("month", F.min("InvoiceDate")).alias("cohort_month"))

# Sub-Table 2: Active purchasing months
purchases_df = df_spark.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .select("CustomerID", F.date_trunc("month", "InvoiceDate").alias("purchase_month")).distinct()

# Join and final index calculation
query_5_ops = cohort_df.join(purchases_df, "CustomerID") \
    .withColumn("acquisition_month", F.date_format("cohort_month", "yyyy-MM")) \
    .withColumn("retention_month_index", F.months_between(F.col("purchase_month"), F.col("cohort_month")).cast("int")) \
    .groupBy("acquisition_month", "retention_month_index") \
    .agg(F.countDistinct("CustomerID").alias("active_customers")) \
    .orderBy("acquisition_month", "retention_month_index")

query_5_ops.show(10)

📊 Executing Query 5 (Cohorts) with PySpark Operators...


  0%|           0/4 Tasks

+-----------------+---------------------+----------------+
|acquisition_month|retention_month_index|active_customers|
+-----------------+---------------------+----------------+
|          2010-12|                    0|             885|
|          2010-12|                    1|             324|
|          2010-12|                    2|             286|
|          2010-12|                    3|             340|
|          2010-12|                    4|             321|
|          2010-12|                    5|             352|
|          2010-12|                    6|             321|
|          2010-12|                    7|             309|
|          2010-12|                    8|             313|
|          2010-12|                    9|             350|
+-----------------+---------------------+----------------+
only showing top 10 rows


## Query 6

### Spark SQL

In [14]:
print("📊 Executing Query 6 (Pareto) with Spark SQL...")

query_6_sql = spark.sql("""
    WITH product_revenue AS (
        SELECT
            StockCode,
            Description AS product_name,
            SUM(Quantity * UnitPrice) AS total_revenue
        FROM retail_data
        GROUP BY StockCode, Description
    ),
    cumulative_data AS (
        SELECT
            product_name,
            total_revenue,
            SUM(total_revenue) OVER(ORDER BY total_revenue DESC) AS cumulative_revenue,
            SUM(total_revenue) OVER() AS global_revenue
        FROM product_revenue
    )
    SELECT
        product_name,
        ROUND(total_revenue, 2) AS total_revenue,
        ROUND(cumulative_revenue, 2) AS cumulative_revenue,
        ROUND((cumulative_revenue / global_revenue) * 100, 2) AS cumulative_percentage
    FROM cumulative_data
    ORDER BY total_revenue DESC
    LIMIT 50
""")
query_6_sql.show(10)

📊 Executing Query 6 (Pareto) with Spark SQL...


  0%|           0/3 Tasks

+--------------------+-------------+------------------+---------------------+
|        product_name|total_revenue|cumulative_revenue|cumulative_percentage|
+--------------------+-------------+------------------+---------------------+
|      DOTCOM POSTAGE|    206248.77|         206248.77|                 1.93|
|REGENCY CAKESTAND...|    174484.74|         380733.51|                 3.57|
|PAPER CRAFT , LIT...|     168469.6|         549203.11|                 5.15|
|WHITE HANGING HEA...|    104340.29|          653543.4|                 6.13|
|       PARTY BUNTING|     99504.33|         753047.73|                 7.06|
|JUMBO BAG RED RET...|     94340.05|         847387.78|                 7.94|
|MEDIUM CERAMIC TO...|     81700.92|          929088.7|                 8.71|
|              Manual|     78110.27|        1007198.97|                 9.44|
|             POSTAGE|     78101.88|        1085300.85|                10.17|
|  RABBIT NIGHT LIGHT|     66964.99|        1152265.84|         

### PySpark Operators

In [15]:
print("📊 Executing Query 6 (Pareto) with PySpark Operators...")

# 1. Aggregate revenue by product
prod_rev_df = df_spark.groupBy("StockCode", "Description") \
    .agg(F.sum(F.col("Quantity") * F.col("UnitPrice")).alias("total_revenue"))

# 2. Define Window Specs
window_spec_cum = Window.orderBy(F.col("total_revenue").desc())
window_spec_glob = Window.partitionBy()

# 3. Calculate running totals and percentages
query_6_ops = prod_rev_df \
    .withColumn("cumulative_revenue", F.sum("total_revenue").over(window_spec_cum)) \
    .withColumn("global_revenue", F.sum("total_revenue").over(window_spec_glob)) \
    .withColumn("cumulative_percentage", F.round((F.col("cumulative_revenue") / F.col("global_revenue")) * 100, 2)) \
    .select(
        F.col("Description").alias("product_name"),
        F.round("total_revenue", 2).alias("total_revenue"),
        F.round("cumulative_revenue", 2).alias("cumulative_revenue"),
        "cumulative_percentage"
    ).orderBy(F.col("total_revenue").desc()).limit(50)

query_6_ops.show(10)

📊 Executing Query 6 (Pareto) with PySpark Operators...


/usr/local/lib/python3.12/dist-packages/pyspark/sql/connect/expressions.py:1091: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  0%|           0/6 Tasks

+--------------------+-------------+------------------+---------------------+
|        product_name|total_revenue|cumulative_revenue|cumulative_percentage|
+--------------------+-------------+------------------+---------------------+
|      DOTCOM POSTAGE|    206248.77|         206248.77|                 1.93|
|REGENCY CAKESTAND...|    174484.74|         380733.51|                 3.57|
|PAPER CRAFT , LIT...|     168469.6|         549203.11|                 5.15|
|WHITE HANGING HEA...|    104340.29|          653543.4|                 6.13|
|       PARTY BUNTING|     99504.33|         753047.73|                 7.06|
|JUMBO BAG RED RET...|     94340.05|         847387.78|                 7.94|
|MEDIUM CERAMIC TO...|     81700.92|          929088.7|                 8.71|
|              Manual|     78110.27|        1007198.97|                 9.44|
|             POSTAGE|     78101.88|        1085300.85|                10.17|
|  RABBIT NIGHT LIGHT|     66964.99|        1152265.84|         

## K-means

In [16]:
print("⚙️ Calculating base RFM metrics and preparing features...")

max_date_val = df_spark.select(F.max(F.to_date("InvoiceDate"))).collect()[0][0]

rfm_df = df_spark.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .groupBy("CustomerID") \
    .agg(
        F.datediff(F.lit(max_date_val), F.max(F.to_date("InvoiceDate"))).cast("double").alias("Recency"),
        F.countDistinct("InvoiceNo").cast("double").alias("Frequency"),
        F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).cast("double").alias("Monetary")
    )

# VectorAssembler
assembler = VectorAssembler(
    inputCols=["Recency", "Frequency", "Monetary"],
    outputCol="raw_features"
)
rfm_assembled = assembler.transform(rfm_df)

# StandardScaler
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(rfm_assembled)
rfm_scaled = scaler_model.transform(rfm_assembled)

print("✅ Features scaled and ready for the model!")
rfm_scaled.select("CustomerID", "scaled_features").show(5, truncate=False)

⚙️ Calculating base RFM metrics and preparing features...


  0%|           0/3 Tasks

  0%|           0/1 Tasks

✅ Features scaled and ready for the model!


  0%|           0/6 Tasks

+----------+----------------------------------------------------------------+
|CustomerID|scaled_features                                                 |
+----------+----------------------------------------------------------------+
|16503     |[0.13938816097729775,-0.035335779443903245,-0.06923133890640312]|
|16339     |[1.9191698842447011,-0.42504750290405535,-0.21629398332460556]  |
|12626     |[-0.6905100582990759,0.6141837596563503,0.5079648998385583]     |
|14423     |[1.2592508183140911,-0.42504750290405535,-0.1990233169409039]   |
|17809     |[-0.7605014743826254,1.0038954831165023,0.3735184633081215]     |
+----------+----------------------------------------------------------------+
only showing top 5 rows


In [17]:
print("🧠 Training K-Means model (k=3)...")

kmeans = KMeans(
    featuresCol="scaled_features",
    predictionCol="Cluster_Segment",
    k=3,
    seed=42
)

kmeans_model = kmeans.fit(rfm_scaled)
predictions = kmeans_model.transform(rfm_scaled)

evaluator = ClusteringEvaluator(
    predictionCol="Cluster_Segment",
    featuresCol="scaled_features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)
silhouette_score = evaluator.evaluate(predictions)
print(f"✅ Model Trained! Silhouette Score: {silhouette_score:.4f}")

print("\n📊 Segment Profiling (Business View)...")
cluster_profile = predictions.groupBy("Cluster_Segment").agg(
    F.count("CustomerID").alias("customer_count"),
    F.round(F.avg("Recency"), 0).alias("avg_recency_days"),
    F.round(F.avg("Frequency"), 1).alias("avg_frequency_orders"),
    F.round(F.avg("Monetary"), 2).alias("avg_monetary_gbp")
).orderBy("Cluster_Segment")

cluster_profile.show()

🧠 Training K-Means model (k=3)...


  0%|           0/4 Tasks

  0%|           0/8 Tasks

✅ Model Trained! Silhouette Score: 0.7133

📊 Segment Profiling (Business View)...


  0%|           0/10 Tasks

+---------------+--------------+----------------+--------------------+----------------+
|Cluster_Segment|customer_count|avg_recency_days|avg_frequency_orders|avg_monetary_gbp|
+---------------+--------------+----------------+--------------------+----------------+
|              0|          3233|            41.0|                 4.9|         2011.56|
|              1|          1091|           246.0|                 1.6|          630.24|
|              2|            14|             6.0|                80.2|       122888.41|
+---------------+--------------+----------------+--------------------+----------------+



## FP-Growth (Frequent Pattern Growth)

In [18]:
print("⚙️ Grouping items by invoice into transaction baskets...")

basket_df = df_spark.filter(F.col("Description").isNotNull()) \
    .groupBy("InvoiceNo") \
    .agg(F.collect_set("Description").alias("items"))

print("🧠 Extracting frequent patterns and training FP-Growth model...")

fp_growth = FPGrowth(
    itemsCol="items",
    minSupport=0.02,
    minConfidence=0.3
)

model = fp_growth.fit(basket_df)
print("✅ Algorithm completed successfully!")

print("📊 Top 10 Frequent Itemsets:")
model.freqItemsets.orderBy(F.col("freq").desc()).show(10, truncate=False)

print("🔮 Top 10 Predictive Association Rules:")
association_rules = model.associationRules.orderBy(F.col("lift").desc())
association_rules.show(10, truncate=False)

⚙️ Grouping items by invoice into transaction baskets...
🧠 Extracting frequent patterns and training FP-Growth model...


  0%|           0/2 Tasks

✅ Algorithm completed successfully!
📊 Top 10 Frequent Itemsets:


  0%|           0/3 Tasks

+------------------------------------+----+
|items                               |freq|
+------------------------------------+----+
|[WHITE HANGING HEART T-LIGHT HOLDER]|2256|
|[JUMBO BAG RED RETROSPOT]           |2089|
|[REGENCY CAKESTAND 3 TIER]          |1988|
|[PARTY BUNTING]                     |1685|
|[LUNCH BAG RED RETROSPOT]           |1564|
|[ASSORTED COLOUR BIRD ORNAMENT]     |1455|
|[SET OF 3 CAKE TINS PANTRY DESIGN ] |1385|
|[PACK OF 72 RETROSPOT CAKE CASES]   |1320|
|[LUNCH BAG  BLACK SKULL.]           |1273|
|[NATURAL SLATE HEART CHALKBOARD ]   |1249|
+------------------------------------+----+
only showing top 10 rows
🔮 Top 10 Predictive Association Rules:


  0%|           0/5 Tasks

+-------------------------------------------------------------------+------------------------------------+------------------+------------------+--------------------+
|antecedent                                                         |consequent                          |confidence        |lift              |support             |
+-------------------------------------------------------------------+------------------------------------+------------------+------------------+--------------------+
|[GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER ]|[PINK REGENCY TEACUP AND SAUCER]    |0.7053455019556715|18.403524469327063|0.027104208416833666|
|[PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER ] |[GREEN REGENCY TEACUP AND SAUCER]   |0.9046822742474916|17.82572378477782 |0.027104208416833666|
|[GREEN REGENCY TEACUP AND SAUCER]                                  |[PINK REGENCY TEACUP AND SAUCER]    |0.6238894373149062|16.27821329255625 |0.031663326653306616|
|[PI

In [19]:
PROJECT_ID = "ccbd-20260619-danieletambone"
DATASET = "cart_dataset"
TARGET_TABLE = f"{PROJECT_ID}.{DATASET}.association_rules"

print(f"🚀 Exporting Association Rules back to BigQuery table: {TARGET_TABLE}...")

# Writing the rules DataFrame back to BigQuery
association_rules.write.format("bigquery") \
  .option("table", TARGET_TABLE) \
  .option("temporaryGcsBucket", "ccbd-20260619-danieletambone-bucket") \
  .mode("overwrite") \
  .save()

print("🎉 Export completed! The recommendation engine is now accessible by Looker Studio.")

🚀 Exporting Association Rules back to BigQuery table: ccbd-20260619-danieletambone.cart_dataset.association_rules...


  0%|           0/16 Tasks

🎉 Export completed! The recommendation engine is now accessible by Looker Studio.


### FP-Growth Results Analysis (Market Basket Insights)

The execution of the FP-Growth algorithm across the historical transaction data has generated highly coherent insights, directly applicable for cross-selling strategies. The algorithm was configured with `minSupport = 0.02` (pattern must appear in at least 2% of all baskets) and `minConfidence = 0.3` (30% conditional probability), isolating the following business insights:

**1. Absolute Frequency Analysis (Top Itemsets)**
The absolute best-selling product is the *"WHITE HANGING HEART T-LIGHT HOLDER"* (present in 2,256 baskets), followed by the *"JUMBO BAG RED RETROSPOT"* and the *"REGENCY CAKESTAND 3 TIER"*. These items act as "Door-Openers": they attract customers and can be strategically used for up-selling promotions.

**2. Association Rules (Predictive Power)**
By sorting the results by **Lift**, extremely strong behavioral patterns emerge that rule out pure statistical coincidence (Lift significantly > 1):
* **The "Teacup" Pattern (Lift = 18.4):** The strongest combination involves the "Regency" tea set. Customers purchasing the Green and Pink variants ("GREEN & ROSES REGENCY TEACUP") have an incredibly high probability (Confidence 70.5%) of completing the set by adding the Pink cup ("PINK"). A Lift of 18.4 indicates that buying the pink variant is 18 times more likely when the other two are present, compared to a random purchase.
* **The "Gardeners" Pattern (Lift = 15.7):** Customers purchasing the "GARDENERS KNEELING PAD CUP OF TEA" have a 72% confidence of also purchasing the "KEEP CALM" variant.

In [20]:
spark.stop()
print("🛑 Session closed. Resources released.")

🛑 Session closed. Resources released.
